In [1]:
pip install numpy Pillow pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import json
import numpy as np
# from sklearn.manifold import TSNE
import pandas as pd
from PIL import Image
arrays = sorted(Path("outputs").glob("*.json"))

In [3]:
annotations = json.loads(Path("./annotation.json").read_text())

* iterate over the items
* for each element in the item list
    * check if there's a visualization_bbox
    * if we find one clip the image, and save in separate folder, give it an incrementing name
    * also add a field to the list,
        * image name, type
        * just to simplify comparison later 

In [8]:
annotations["291"]

[{'image_id': 0,
  'file_name': '0291_00.png',
  'page': 1,
  'dpi': 300,
  'bbox': [0.10851926977687627,
   0.06183574879227053,
   0.49813200498132004,
   0.2927536231884058],
  'visualization_bbox': {'flow_diagram': [[39.52336448598135,
     73.71028037383178,
     883.8411214953272,
     717.0]]},
  'nums_of_visualizations': {'flow_diagram': 1},
  'caption': 'Figure 1: VRMosaic Architecture'},
 {'image_id': 1,
  'file_name': '0291_01.png',
  'page': 2,
  'dpi': 300,
  'bbox': [0.5087691703702959,
   0.09572941153185632,
   0.8691683569979716,
   0.30486001577287064],
  'visualization_bbox': {},
  'nums_of_visualizations': {},
  'caption': 'Figure 3 shows VRMosaic with the 2D schematic'},
 {'image_id': 2,
  'file_name': '0291_02.png',
  'page': 2,
  'dpi': 300,
  'bbox': [0.5121703853955375,
   0.36321963722397477,
   0.8752535496957403,
   0.5698442429022083],
  'visualization_bbox': {},
  'nums_of_visualizations': {},
  'caption': 'Figure 3 shows VRMosaic with the 2D schematic'},


In [4]:
from PIL import Image

In [5]:
out = Path("subfigures")
out.mkdir(exist_ok=True)

In [6]:
lim = 10
rawtable = []
errors = []
import tqdm
for k,v in tqdm.tqdm(list(annotations.items())):
    try:
        # lim-=1
        # if lim <0:
        #     break
        # now iterate over the values array
        for element in v:
            vis_counter=0
            vbox = element.get("visualization_bbox",-1)
            if vbox!=-1:
                # load the image
                name = element.get("file_name")
                # print(name)
                folder = name.split("_")[0]
                if folder[0]=="0":
                    folder=folder[1:]
                image_name = name.split("_")[1]
                if image_name[0] == "0":
                    image_name =  image_name[1:]
                image_name=Path(image_name)
                # im = Image.open(Path(f"images/{folder}/{image_name}"))
                for chart,bboxes in vbox.items():
                    for bbox in bboxes:
                        # clip oout the parts we need
                        # print(bbox)
                        # sub = im.crop(bbox)
                        out_name = f"{folder:04}_{image_name.stem:04}_{vis_counter}.png"
                        rawtable.append({"image_name":out_name,"chart_type":chart})
                        # sub.save(f"{out}/{out_name}")
                        vis_counter+=1
    except Exception as e:
        errors.append([k,e])
        

100%|█████████████████████████████████████| 1395/1395 [00:00<00:00, 7145.10it/s]


In [7]:
df = pd.DataFrame(rawtable)

In [8]:
df

,image_name,chart_type
0,1021_0000_0.png,line_chart
1,1021_2000_0.png,line_chart
2,1021_3000_0.png,line_chart
3,1021_3000_1.png,line_chart
4,1021_3000_2.png,line_chart
...,...,...
35090,9370_0000_0.png,graph
35091,9370_1000_0.png,table
35092,9370_2000_0.png,graph
35093,9370_4000_0.png,bar_chart


In [9]:
df.to_csv("visimages_annotations.csv")

In [20]:
[k for k,v in annotations.items()].index("2076")

508

mostly unnec code from last time
```
classes = ('flow_diagram', 'scatterplot', 'bar_chart', 'graph', 'treemap', 'table', 'line_chart', 'tree', 'small_multiple', 'heatmap', 'matrix', 'map', 'pie_chart', 'sankey_diagram', 'area_chart', 'proportional_area_chart', 'glyph_based', 'stripe_graph', 'parallel_coordinate', 'sunburst_icicle', 'unit_visualization', 'polar_plot', 'error_bar', 'box_plot', 'sector_chart', 'word_cloud', 'donut_chart', 'hierarchical_edge_bundling', 'chord_diagram', 'storyline')

print(arrays[0])
fname = arrays[0]
# testing with specific multi plot values
fname = Path("outputs/array_fig-2340671844-Figure50-1.json")
figure_name = f"{fname.parent}/{fname.stem.split('_')[1]}.png"
print(figure_name)
figure_vectors = []
for fname in arrays:
    json_data = json.loads(Path(fname).read_text())
    figure_name = f"{fname.parent}/result_{fname.stem.split('_')[1]}.jpg"
    
    vis_dict = {}
    vis_dict["json"] = str(fname)
    vis_dict["image"] = figure_name
    charts={}
    for i,result in enumerate(json_data):
        # might be more than one element here
        ctype = classes[i]
        
        for detection in result:
            chart_record = charts.get(ctype,[])
            chart_record.append(detection)
            charts[ctype] = chart_record
    vis_dict["charts"]= charts
    figure_vectors.append(vis_dict)
```

In [4]:
len(figure_vectors)

10000

In [5]:
import tqdm

In [6]:
import uuid
# make a script that reads the figures and then crops out part using bounding box
def crop_part(im_path,bbox,out_pth):
    im = Image.open(str(im_path))
    res = im.crop(bbox)
    res.save(f"{out_pth}/{uuid.uuid4()}.png")
threshold = .5
for e in tqdm.tqdm(figure_vectors):
    #e = figure_vectors[0]
    annotated_path = Path(e["image"])
    stem = annotated_path.stem.split("_")[1]
    im_path = Path(f"/xdisk/chrisreidy/baylyd/vis_sieve/vis-sieve/pdf_grabbing/pdf_symlinks/{stem}.png")
    charts = e["charts"]
    out_path = Path("sub_figures_all")
    out_path.mkdir(exist_ok=True)
    for chart_name in charts:
        parts = charts[chart_name]
        for part in parts:
            conf = part[4]
            if conf > threshold:
                bbox = part[:4]
                crop_part(im_path,bbox,out_path)


100%|█████████████████████████████████████| 10000/10000 [13:43<00:00, 12.15it/s]


In [13]:
Path(e["image"]).stem.split("_")[1]

'fig-4206307734-Figure6-1'

In [9]:
list(Path("/xdisk/chrisreidy/baylyd/vis_sieve/vis-sieve/pdf_grabbing/pdf_symlinks").iterdir())

[PosixPath('/xdisk/chrisreidy/baylyd/vis_sieve/vis-sieve/pdf_grabbing/pdf_symlinks/fig-4225268984-Figure6-1.png'),
 PosixPath('/xdisk/chrisreidy/baylyd/vis_sieve/vis-sieve/pdf_grabbing/pdf_symlinks/fig-3205096573-Figure4-1.png'),
 PosixPath('/xdisk/chrisreidy/baylyd/vis_sieve/vis-sieve/pdf_grabbing/pdf_symlinks/fig-4220746662-Figure9-1.png'),
 PosixPath('/xdisk/chrisreidy/baylyd/vis_sieve/vis-sieve/pdf_grabbing/pdf_symlinks/fig-4213312616-Table5-1.png'),
 PosixPath('/xdisk/chrisreidy/baylyd/vis_sieve/vis-sieve/pdf_grabbing/pdf_symlinks/fig-4312772811-TableI-1.png'),
 PosixPath('/xdisk/chrisreidy/baylyd/vis_sieve/vis-sieve/pdf_grabbing/pdf_symlinks/data-4307328413.json'),
 PosixPath('/xdisk/chrisreidy/baylyd/vis_sieve/vis-sieve/pdf_grabbing/pdf_symlinks/fig-4281635365-FigureB.2-1.png'),
 PosixPath('/xdisk/chrisreidy/baylyd/vis_sieve/vis-sieve/pdf_grabbing/pdf_symlinks/4310973913.pdf'),
 PosixPath('/xdisk/chrisreidy/baylyd/vis_sieve/vis-sieve/pdf_grabbing/pdf_symlinks/fig-4386433108-Figu